In [5]:
from catboost import CatBoostClassifier, Pool, sum_models
import numpy as np
from torch.utils.data import DataLoader, Dataset
import pandas
from scipy.interpolate import interp1d
import torch
from tqdm import tqdm
import sklearn

In [17]:
def load(name):
    with open(name, 'rb') as f:
        return np.load(f)

def interpolate_data(data, begin_index, end_index, new_length=1300):
    old_indices = np.linspace(0, len(data)-1, num=len(data))
    new_indices = np.linspace(0, len(data)-1, num=new_length)
    interpolator = interp1d(old_indices, data)
    new_data = interpolator(new_indices)
    fft = np.fft.fft(new_data).flatten()
    return new_data[begin_index:end_index], fft[begin_index:end_index].real

class BoostDataset(Dataset):
    def __init__(self):
        self.data = pandas.read_csv('../peaks_dataset_norm.csv').to_numpy()
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        data = load(self.data[idx][0])
        X = []
        fft = []
        for i in data:
            tmp_x, tmp_fft = interpolate_data(i, int(self.data[idx][2]), int(self.data[idx][3]))
            X.append(tmp_x)
            fft.append(tmp_fft)
            
        data = X
        for i in fft:
            data.append(i)
        data = np.array(data)
        return data, int(self.data[idx][1])

In [21]:
dataset = BoostDataset()
train_size = int(0.9 * len(dataset))
val_size = int(0.04 * len(dataset))
print(train_size)
print(val_size)
test_size = len(dataset) - train_size - val_size

train_data, val_data, test_data = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_loader = torch.utils.data.DataLoader(train_data, batch_size=1768278//2, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=78590, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=512, shuffle=True)


def validate():
    for x, y in val_loader:
        x = x.numpy().reshape(x.shape[0], -1)
        y = y.numpy()
        return x, y
    

1768278
78590


In [22]:
from sklearn.metrics import recall_score, f1_score, precision_score
epoch = 5
models = []
params = {
    'task_type': 'CPU',
    'iterations': 2500,
    'learning_rate': 0.08,
    'depth': 7,
    'loss_function': 'Logloss',
    'eval_metric': 'F1'
}
model = CatBoostClassifier(**params)
x_val, y_val = validate()
val = Pool(data=x_val, label=y_val)
i = 0
for  x, y in tqdm(train_loader):
    x = x.numpy().reshape(x.shape[0], -1)
    y = y.numpy()
    train_dataset = Pool(data=x, label=y)
    if i == 0:
        model.fit(train_dataset, verbose=1, eval_set=val)
    else:
        model.fit(train_dataset, verbose=1, eval_set=val, init_model='model.cbm')
    i+=1
    model.save_model('model.cbm')
    
    

f1, rec, pres = [], [], []

for x, y in val_loader:
    x = x.numpy().reshape(x.shape[0], -1)
    y = y.numpy()
    y_pred = model.predict(Pool(data=x))
    metric = model.eval_metrics(Pool(x, y), ['F1'])
    print(np.mean(metric['F1']))
    rec.append(recall_score(y_pred, y))
    pres.append(precision_score(y_pred, y))
    f1.append(f1_score(y_pred, y))

print(f'f1 = {np.mean(f1)}, precision = {np.mean(pres)}, recall = {np.mean(rec)}')  
    
    


    






  0%|          | 0/2 [09:49<?, ?it/s]


KeyboardInterrupt: 

In [16]:
dataset = pandas.read_csv('../peaks_dataset.csv').to_numpy()
true_val = []
false_val = []
for i in dataset:
    if i[1] == 0:
        false_val.append(i)
    else:
        true_val.append(i)
        
false_val = false_val[0:len(true_val)]

res = []
for i in range(len(true_val)):
    res.append(false_val[i])
    res.append(true_val[i])
    
df = pandas.DataFrame(res)
df.to_csv('../peaks_dataset_norm.csv', index=False)
